In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import py4cytoscape as p4c


# =========================
# SETUP
# =========================

BASE = Path.home() / "Downloads" / "Cyto"

for d in [
    "01_raw_qc", "02_clean_data", "03_mapping",
    "04_networks_global", "05_networks_modules",
    "06_hubs_python", "07_hubs_cytohubba_proxy",
    "08_concordance", "09_plots",
    "10_cytoscape_session", "11_logs"
]:
    (BASE / d).mkdir(parents=True, exist_ok=True)


# =========================
# LOGGING
# =========================

log_file = BASE / "11_logs" / "log.txt"
err_file = BASE / "11_logs" / "error.txt"

def log(msg):
    with open(log_file, "a") as f:
        f.write(str(msg) + "\n")

def fail(msg):
    with open(err_file, "a") as f:
        f.write(str(msg) + "\n")
    raise ValueError(msg)


# =========================
# CLEANING
# =========================

def clean(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].str.replace('"', '', regex=False)\
                           .str.replace("'", '', regex=False)\
                           .str.strip()
    return df


# =========================
# LOAD
# =========================

def load(nodes_path, edges_path, biomart_path):
    nodes = clean(pd.read_csv(nodes_path, sep=None, engine="python"))
    edges = clean(pd.read_csv(edges_path))
    biomart = clean(pd.read_csv(biomart_path))
    return nodes, edges, biomart


# =========================
# GENE NORMALIZATION (CRITICAL CORE)
# =========================

def normalize_gene(x):
    """
    Handles:
    - ENSG versions (ENSG000..1 → ENSG000..)
    - raw symbols
    - whitespace
    """
    x = str(x).strip()

    # remove Ensembl version suffix
    if x.startswith("ENSG") and "." in x:
        x = x.split(".")[0]

    return x

In [2]:
def build_map(biomart):

    biomart = biomart.dropna(subset=["hgnc_symbol"])
    biomart = biomart[biomart["hgnc_symbol"] != ""]

    biomart["ensembl_gene_id"] = biomart["ensembl_gene_id"].astype(str).apply(normalize_gene)

    # remove duplicates (keep first valid mapping)
    biomart = biomart.drop_duplicates(subset="ensembl_gene_id")

    return dict(zip(biomart["ensembl_gene_id"], biomart["hgnc_symbol"]))

In [3]:
def standardize(nodes, edges, mp):

    nodes = nodes.copy()
    edges = edges.copy()

    # normalize everything first
    nodes["gene_raw"] = nodes["gene"].astype(str)
    nodes["gene_clean"] = nodes["gene_raw"].apply(normalize_gene)

    edges["Gene1"] = edges["Gene1"].astype(str).apply(normalize_gene)
    edges["Gene2"] = edges["Gene2"].astype(str).apply(normalize_gene)

    # detect ENSG
    nodes["is_ensembl"] = nodes["gene_clean"].str.startswith("ENSG")

    # mapping attempt
    nodes["gene_mapped"] = nodes["gene_clean"].map(mp)

    # FINAL ID (biologically safe fallback logic)
    nodes["id"] = np.where(
        nodes["gene_mapped"].notna(),
        nodes["gene_mapped"],     # HGNC if available
        nodes["gene_clean"]       # fallback: original cleaned ID
    )

    # edges mapping (same logic)
    def map_edge(x):
        x = normalize_gene(x)
        return mp.get(x, x)

    edges["Gene1"] = edges["Gene1"].apply(map_edge)
    edges["Gene2"] = edges["Gene2"].apply(map_edge)

    return nodes, edges

In [4]:
def mapping_qc(nodes, mp):

    ensembl = nodes[nodes["is_ensembl"]].copy()
    ensembl["mapped"] = ensembl["gene_clean"].map(mp)

    unmapped = ensembl[ensembl["mapped"].isna()]

    print("\n=== MAPPING QC REPORT ===")
    print(f"Total genes: {len(nodes)}")
    print(f"Ensembl genes: {len(ensembl)}")
    print(f"Mapped Ensembl: {len(ensembl) - len(unmapped)}")
    print(f"Unmapped Ensembl: {len(unmapped)}")
    print(f"Mapping rate: {(len(ensembl) - len(unmapped)) / max(len(ensembl),1):.3f}")

    print("\nTop unmapped genes:")
    print(unmapped["gene_clean"].head(15).tolist())

    unmapped.to_csv(BASE / "01_raw_qc/unmapped_ensembl.csv", index=False)

    return unmapped

In [5]:
def qc(nodes, edges):

    edges = edges[edges["Gene1"] != edges["Gene2"]].copy()

    valid_nodes = set(nodes["id"])

    missing = (set(edges["Gene1"]) | set(edges["Gene2"])) - valid_nodes

    log(f"Missing nodes in edges: {len(missing)}")

    return edges

In [6]:
def hubs(nodes, edges):

    deg = pd.concat([edges["Gene1"], edges["Gene2"]]).value_counts()

    nodes = nodes.copy()
    nodes["degree"] = nodes["id"].map(deg).fillna(0)

    nodes["kWithin_norm"] = nodes.groupby("module")["kWithin"].transform(
        lambda x: x / (x.max() if x.max() > 0 else 1)
    )

    nodes["degree_norm"] = (
        (nodes["degree"] - nodes["degree"].min()) /
        (nodes["degree"].max() - nodes["degree"].min() + 1e-9)
    )

    nodes["hybrid"] = 0.7 * nodes["kWithin_norm"] + 0.3 * nodes["degree_norm"]

    return nodes

In [7]:
def cytohubba_proxy(edges):

    graph = {}

    for _, r in edges.iterrows():
        graph.setdefault(r["Gene1"], set()).add(r["Gene2"])
        graph.setdefault(r["Gene2"], set()).add(r["Gene1"])

    mcc = {}

    for n in graph:
        neigh = list(graph[n])
        score = 0

        for i in range(len(neigh)):
            for j in range(i + 1, len(neigh)):
                if neigh[j] in graph.get(neigh[i], set()):
                    score += 1

        mcc[n] = score

    return pd.DataFrame({
        "gene_symbol": list(mcc.keys()),
        "MCC_proxy": list(mcc.values())
    })

In [8]:
def module_networks(nodes, edges):

    for m in nodes["module"].dropna().unique():

        n = nodes[nodes["module"] == m]
        valid = set(n["id"])

        e = edges[
            edges["Gene1"].isin(valid) &
            edges["Gene2"].isin(valid)
        ]

        if len(e) == 0:
            log(f"EMPTY MODULE {m}")
            continue

        path = BASE / "05_networks_modules" / str(m)
        path.mkdir(parents=True, exist_ok=True)

        n.to_csv(path / "nodes.csv", index=False)
        e.to_csv(path / "edges.csv", index=False)

In [9]:
'''
def cytoscape(nodes, edges):

    p4c.cytoscape_ping()
    p4c.delete_all_networks()

    nodes_cyto = nodes.copy().drop_duplicates(subset="id")
    edges_cyto = edges.copy()

    valid = set(nodes_cyto["id"])

    edges_cyto = edges_cyto[
        edges_cyto["Gene1"].isin(valid) &
        edges_cyto["Gene2"].isin(valid)
    ].copy()

    if len(edges_cyto) == 0:
        fail("No valid edges after filtering")

    net_suid = p4c.create_network_from_data_frames(
        nodes=nodes_cyto,
        edges=edges_cyto,
        title="WGCNA Network",
        collection="Cyto Pipeline",
        node_id_list="id",
        source_id_list="Gene1",
        target_id_list="Gene2"
    )

    p4c.load_table_data(
        data=nodes_cyto,
        data_key_column="id",
        table="node",
        table_key_column="name"
    )

    p4c.layout_network("force-directed", network=net_suid)

    p4c.save_session(str(BASE / "10_cytoscape_session" / "session.cys"))
    '''



import pandas as pd
import numpy as np
from pathlib import Path
import py4cytoscape as p4c


# =========================
# SETUP
# =========================

BASE = Path.home() / "Downloads" / "Cyto"


# =========================
# LOGGING (optional but safe)
# =========================

def log(msg):
    print(msg)


def fail(msg):
    raise ValueError(msg)


# =========================
# CORE ASSUMPTION
# =========================
# nodes must contain:
# - id (standardized gene name)
# - module (WGCNA module label)
#
# edges must contain:
# - Gene1
# - Gene2


# =========================
# CYTOSCAPE MULTI-MODULE EXPORT
# =========================

def cytoscape_modules(nodes, edges):

    p4c.cytoscape_ping()
    p4c.delete_all_networks()

    # =========================
    # CLEAN INPUT
    # =========================
    nodes = nodes.copy()
    edges = edges.copy()

    nodes = nodes.dropna(subset=["module"])

    nodes["module"] = nodes["module"].astype(str)
    nodes["id"] = nodes["id"].astype(str)

    edges["Gene1"] = edges["Gene1"].astype(str)
    edges["Gene2"] = edges["Gene2"].astype(str)

    # =========================
    # COLOR PALETTE
    # =========================
    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
        "#3182bd", "#e6550e", "#31a354", "#756bb1", "#636363",
        "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
        "#ffff33", "#a65628", "#f781bf", "#999999", "#66c2a5",
        "#fc8d62", "#8da0cb", "#e78ac3", "#a6d854", "#ffd92f",
        "#e5c494", "#b3b3b3", "#1b9e77", "#d95f02", "#7570b3"
    ]

    modules = sorted(nodes["module"].unique())

    if len(modules) > len(palette):
        raise ValueError("Too many modules for 40-color palette")

    # =========================
    # CREATE ONE NETWORK PER MODULE
    # =========================
    for i, m in enumerate(modules):

        print(f"Building module: {m}")

        # ---- subset nodes ----
        n = nodes[nodes["module"] == m].copy()

        if n.empty:
            continue

        # assign module color (IMPORTANT FIX)
        n["color"] = palette[i]

        valid_nodes = set(n["id"])

        # ---- subset edges ----
        e = edges[
            edges["Gene1"].isin(valid_nodes) &
            edges["Gene2"].isin(valid_nodes)
        ].copy()

        e = e[e["Gene1"] != e["Gene2"]]

        if e.empty:
            print(f"Skipping empty module: {m}")
            continue

        # =========================
        # CREATE NETWORK
        # =========================
        net_suid = p4c.create_network_from_data_frames(
            nodes=n,
            edges=e,
            title=f"Module_{m}",
            collection="WGCNA_Modules",
            node_id_list="id",
            source_id_list="Gene1",
            target_id_list="Gene2"
        )

        # =========================
        # APPLY COLOR (CORRECT METHOD)
        # =========================
        p4c.set_node_color_mapping(
            table_column="color",
            mapping_type="passthrough"
        )

        # =========================
        # LAYOUT
        # =========================
        p4c.layout_network("force-directed", network=net_suid)

    # =========================
    # SAVE SESSION
    # =========================
    session_path = str(BASE / "10_cytoscape_session" / "module_networks.cys")
    p4c.save_session(filename=session_path)

    print(f"Saved Cytoscape session: {session_path}")

In [10]:
def run(nodes_path, edges_path, biomart_path):

    log("START")

    nodes, edges, biomart = load(nodes_path, edges_path, biomart_path)

    mp = build_map(biomart)

    nodes, edges = standardize(nodes, edges, mp)

    mapping_qc(nodes, mp)

    edges = qc(nodes, edges)

    nodes.to_csv(BASE / "02_clean_data/nodes.csv", index=False)
    edges.to_csv(BASE / "02_clean_data/edges.csv", index=False)

    nodes = hubs(nodes, edges)
    nodes.to_csv(BASE / "06_hubs_python/hubs.csv", index=False)

    proxy = cytohubba_proxy(edges)
    proxy.to_csv(BASE / "07_hubs_cytohubba_proxy/mcc_proxy.csv", index=False)

    '''module_networks(nodes, edges)

    cytoscape(nodes, edges)'''

    module_networks(nodes, edges)   # optional file export
    cytoscape_modules(nodes, edges)  # actual Cytoscape visualization

    log("DONE")

    # =========================
# RUN
# =========================

run(
    str(Path.home() / "Downloads" / "hub_genes_per_module (1).txt"),
    str(Path.home() / "Downloads" / "cytoscape_latest_edges.csv"),
    str(Path.home() / "Downloads" / "Cyto" / "03_mapping" / "biomart_mapping.csv")
)

START

=== MAPPING QC REPORT ===
Total genes: 560
Ensembl genes: 190
Mapped Ensembl: 77
Unmapped Ensembl: 113
Mapping rate: 0.405

Top unmapped genes:
['ENSG00000261442', 'ENSG00000274910', 'ENSG00000254671', 'ENSG00000267063', 'ENSG00000272583', 'ENSG00000279940', 'ENSG00000283108', 'ENSG00000280177', 'ENSG00000250734', 'ENSG00000258930', 'ENSG00000261655', 'ENSG00000229313', 'ENSG00000255026', 'ENSG00000236842', 'ENSG00000268777']
Missing nodes in edges: 61
EMPTY MODULE black
EMPTY MODULE cyan
EMPTY MODULE green
EMPTY MODULE greenyellow
EMPTY MODULE magenta
EMPTY MODULE pink
EMPTY MODULE purple
EMPTY MODULE salmon
EMPTY MODULE tan
EMPTY MODULE yellow
You are connected to Cytoscape!
Building module: black
Skipping empty module: black
Building module: blue
Applying default style...
Applying preferred layout
style_name not specified, so updating "default" style.
Building module: brown
Applying default style...
Applying preferred layout
style_name not specified, so updating "default" sty